# Diversity test — generation phase (Colab A100)

This notebook runs **only the GPU half** of the model test: it samples N candidate eBPF
programs from the SFT-v2 model, encodes them, and uploads the result to HuggingFace.

The **validation half** (KCOV coverage) runs on your WSL machine where the VM lives — see
the last cell for the exact commands. Colab cannot run the KCOV VM.

**Run the cells top to bottom.** The only cell you may need to edit is the **Config** cell.

In [ ]:
# Cell 1 — Mount Google Drive (the trained adapter lives here)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 2 — Clone repo on first run, pull on restarts (idempotent). Always uses main.
import os
from google.colab import userdata

token = userdata.get('GITHUB_TOKEN')
repo_dir = '/content/ebpf-fuzzing-thesis'

if not os.path.exists(repo_dir):
    ret = os.system(f'git clone https://{token}@github.com/Strhata/ebpf-fuzzing-thesis.git {repo_dir}')
    assert ret == 0, 'git clone failed'

os.chdir(repo_dir)
assert os.system('git checkout main') == 0, 'git checkout failed'
assert os.system('git pull origin main') == 0, 'git pull failed'
print(f'Working directory: {os.getcwd()}')
os.system('git log --oneline -1')

In [ ]:
# Cell 3 — Install dependencies (~5-10 min on a fresh runtime; instant if already installed)
import subprocess
subprocess.run(['pip', 'install', '-q', '-r', 'ml/requirements_colab.txt'], check=True)
print('[+] deps installed')

In [ ]:
# Cell 4 — Config (the ONLY cell you may need to edit). Re-run after any edit, then run Cell 5.
import os, glob

# --- model: base + trained adapter (bf16 adapter-on-base, no merge step) ---
BASE_MODEL = 'Qwen/Qwen2.5-Coder-1.5B'
ADAPTER    = '/content/drive/MyDrive/sft-1epoch-v2/sft_adapter'

# --- sampling knobs ---
N            = 1000     # number of candidate programs to generate
TEMPERATURE  = 1.0
TOP_P        = 0.95
BATCH_SIZE   = 256      # A100 40GB headroom for a 1.5B model; drop to 128 if OOM
MAX_NEW_TOK  = 512
SEED         = 42

# --- where the artifact goes (written locally on Colab; transferred off in Cell 6, no HF token) ---
RUN_NAME      = 'sft-v2-n1000-seed42'
OUT_LOCAL     = f'benchmarks/diversity/candidates/{RUN_NAME}.jsonl'
DRIVE_BACKUP  = f'/content/drive/MyDrive/{RUN_NAME}.jsonl'

# --- sanity check: the adapter must actually be where ADAPTER points ---
if not os.path.exists(os.path.join(ADAPTER, 'adapter_config.json')):
    print(f'[!] No adapter_config.json under {ADAPTER!r}.')
    print('[!] Searching your Drive for adapters — set ADAPTER to one of these and re-run:')
    for p in glob.glob('/content/drive/MyDrive/**/adapter_config.json', recursive=True):
        print('    ', os.path.dirname(p))
    raise SystemExit('Fix ADAPTER above, then re-run this cell.')

print(f'[+] adapter OK: {ADAPTER}')
print(f'[+] will generate N={N} (batch={BATCH_SIZE}) -> {OUT_LOCAL}')

In [ ]:
# Cell 5 — Generate (the main step; uses the A100). Writes candidates locally; transfer in Cell 6.
import subprocess

cmd = (
    f'python -u tools/diversity_sample.py generate'
    f' --base {BASE_MODEL}'
    f' --adapter {ADAPTER}'
    f' --n {N}'
    f' --temperature {TEMPERATURE}'
    f' --top-p {TOP_P}'
    f' --batch-size {BATCH_SIZE}'
    f' --max-new-tokens {MAX_NEW_TOK}'
    f' --seed {SEED}'
    f' --out {OUT_LOCAL}'
)
print('LAUNCH:', cmd, '\n')
# Stream the child's pipe through Python print so Colab actually displays it (stdout+stderr merged).
proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print(f'\n[exit {proc.returncode}]')
if proc.returncode == 0:
    print(f'[+] DONE. Candidates written to {OUT_LOCAL}')
    print('[+] Now run Cell 6 to copy the file off Colab (Drive backup + browser download).')
else:
    print('[!] generate exited non-zero — the traceback is just above')

## Validation phase — run on your WSL machine (where the KCOV VM lives)

Colab's job is done once Cell 6 has backed up to Drive and downloaded the file. The coverage
numbers come from the KCOV VM, which only exists on your local machine.

Cell 6 downloaded the candidates into your **Windows Downloads** folder. From a WSL terminal:

```bash
cd /home/stefano-u/tesi/ebpf-fuzzing-thesis
git checkout main && git pull origin main

# 1. copy in the file Cell 6 downloaded (find <YourWindowsName> with: ls /mnt/c/Users/)
mkdir -p benchmarks/diversity/candidates
cp /mnt/c/Users/<YourWindowsName>/Downloads/sft-v2-n1000-seed42.jsonl \
   benchmarks/diversity/candidates/

# 2. boot the KCOV VM (leave running; open a 2nd terminal for step 3)
./fuzzing/run_eval_vm.sh

# 3. validate -> diversity KPIs
pixi run python tools/diversity_sample.py validate \
    --candidates benchmarks/diversity/candidates/sft-v2-n1000-seed42.jsonl \
    --out benchmarks/diversity/sft-v2-n1000-seed42.json

# 4. read the result
cat benchmarks/diversity/sft-v2-n1000-seed42.json
```

The two numbers that matter for the thesis: **`total_unique_pcs`** (diversity) and
**`novelty_score`** (anti-clustering).

## Validation phase — run on your WSL machine (where the KCOV VM lives)

Colab is done once Cell 5 prints **DONE**. The coverage numbers come from the KCOV VM,
which only exists on your local machine. In a WSL terminal, from the repo root:

```bash
cd /home/stefano-u/tesi/ebpf-fuzzing-thesis
git checkout main && git pull origin main

# 1. download the candidates Colab just uploaded
pixi run huggingface-cli download Strhata/ebpf-corpus \
    candidates/sft-v2-n1000-seed42.jsonl \
    --repo-type dataset --local-dir benchmarks/diversity

# 2. boot the KCOV VM (leave running; open a 2nd terminal for step 3)
./fuzzing/run_eval_vm.sh

# 3. validate -> diversity KPIs
pixi run python tools/diversity_sample.py validate \
    --candidates benchmarks/diversity/candidates/sft-v2-n1000-seed42.jsonl \
    --out benchmarks/diversity/sft-v2-n1000-seed42.json

# 4. read the result
cat benchmarks/diversity/sft-v2-n1000-seed42.json
```

The two numbers that matter for the thesis: **`total_unique_pcs`** (diversity) and
**`novelty_score`** (anti-clustering).